# Wykrywanie naczyń krwionośnych dna siatkówki oka

**Skład grupy:** *[wpisz imiona]*
**Baza danych:** CHASE_DB1 (28 obrazów 999×960 px, dwie maski eksperckie)

---
Realizowane wymagania:
- **3.0** — przetwarzanie obrazu (filtr Frangiego)
- **4.0** — klasyfikator ML (Random Forest, wycinki 5×5 px)
- **5.0** — głęboka sieć neuronowa (CNN, wycinki 32×32 px)


In [ ]:
# ── instalacja potrzebnych bibliotek ──────────────────────────────────────────
# odpal ten blok raz, potem możesz zakomentować jeśli wszystko już masz
# (tensorflow może trochę trwać)
!pip install numpy pillow scikit-image scikit-learn pandas matplotlib tensorflow


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from skimage import morphology, exposure
from skimage.filters import frangi, apply_hysteresis_threshold, gaussian
from skimage.util import view_as_windows
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

warnings.filterwarnings('ignore')

# ścieżki do danych – zmień jeśli folder jest gdzie indziej
DATA_DIR  = Path('CHASE_DB1')
IMG_DIR   = DATA_DIR / 'images'
MASK_DIR  = DATA_DIR / 'masks_1st'

image_ids = sorted([p.stem for p in IMG_DIR.glob('*.jpg')])
print(f"Znaleziono {len(image_ids)} obrazów")


In [ ]:
# ── funkcje pomocnicze używane przez wszystkie trzy metody ────────────────────

def wczytaj(img_id):
    img  = np.array(Image.open(IMG_DIR / f"{img_id}.jpg"))
    maska = np.array(Image.open(MASK_DIR / f"{img_id}_1stHO.png").convert("L")) > 127
    return img, maska

def maska_fov(img_rgb, prog=20):
    fov = np.mean(img_rgb, axis=2) > prog
    return morphology.binary_erosion(fov, morphology.disk(5))

def preprocessing(img_rgb):
    # kanał zielony -> inwersja -> rozmycie -> CLAHE
    g = img_rgb[..., 1].astype(np.float32) / 255.0
    inv = 1.0 - g
    wygladzony = gaussian(inv, sigma=1.5)
    return exposure.equalize_adapthist(wygladzony, clip_limit=0.015)

def ocen(pred, gt, fov):
    y_true = gt[fov].astype(int).ravel()
    y_pred = pred[fov].astype(int).ravel()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    acc  = (tp + tn) / (tp + tn + fp + fn)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {"accuracy": acc, "sensitivity": sens, "specificity": spec,
            "g_mean": np.sqrt(sens * spec), "balanced_acc": (sens + spec) / 2}

def nakladka(rgb, pred, gt, fov):
    out = rgb.copy()
    pm, gm = pred & fov, gt & fov
    out[pm & gm]  = [0, 255, 0]   # TP – zielony
    out[pm & ~gm] = [255, 0, 0]   # FP – czerwony
    out[~pm & gm] = [0, 0, 255]   # FN – niebieski
    return out

print("Funkcje pomocnicze gotowe")


---
## 1. Przetwarzanie obrazu — wymagania na 3.0

Pipeline:
1. Wybranie **kanału zielonego** (najlepszy kontrast naczyń)
2. **Inwersja** (żeby Frangi szukał jasnych struktur)
3. **Gaussian blur** σ=1.5 (usuwamy drobny szum i teksturę naczyniówki – bez tego Frangi generuje mnóstwo FP)
4. **CLAHE** clip_limit=0.015 (lokalne wyrównanie kontrastu)
5. **Filtr Frangiego** σ ∈ {1..5} (wykrywanie struktur tubularnych)
6. **Hysteresis thresholding** (percentyle 85/96) + morfologia + maska FOV


In [ ]:
def segment_vessels(img_rgb):
    fov  = maska_fov(img_rgb)
    prep = preprocessing(img_rgb)
    vess = frangi(prep, sigmas=range(1, 6), black_ridges=False)

    low  = np.percentile(vess[fov], 85)
    high = np.percentile(vess[fov], 96)

    binary = apply_hysteresis_threshold(vess, low, high)
    binary = morphology.binary_closing(binary, morphology.disk(2))
    binary = morphology.remove_small_objects(binary, min_size=100)
    binary = binary & fov
    return binary, fov, vess

# szybki test
img_test, gt_test = wczytaj("Image_01L")
pred_test, fov_test, _ = segment_vessels(img_test)
m = ocen(pred_test, gt_test, fov_test)
print(f"Testowy obraz -> acc={m['accuracy']:.3f}  sens={m['sensitivity']:.3f}  spec={m['specificity']:.3f}")


In [ ]:
# podział train/test – taki sam w całym projekcie
test_ids  = [i for i in image_ids if any(i.startswith(p)
              for p in ("Image_11","Image_12","Image_13","Image_14"))]
train_ids = [i for i in image_ids if i not in test_ids]
print(f"Train: {len(train_ids)} obrazów  |  Test: {len(test_ids)} obrazów")

# ewaluacja baseline na zbiorze testowym
wyniki_bl = []
cache_bl  = {}  # zapamiętaj wyniki do wizualizacji

for img_id in test_ids:
    img, gt = wczytaj(img_id)
    pred, fov, vess = segment_vessels(img)
    cache_bl[img_id] = (img, gt, pred, fov, vess)
    m = ocen(pred, gt, fov)
    m['image'] = img_id
    wyniki_bl.append(m)
    print(f"  {img_id}: acc={m['accuracy']:.4f}  sens={m['sensitivity']:.4f}  spec={m['specificity']:.4f}")

cols = ["accuracy", "sensitivity", "specificity", "g_mean", "balanced_acc"]
df_bl = pd.DataFrame(wyniki_bl).set_index('image')[cols].round(4)
df_bl.loc["SREDNIA"] = df_bl.mean()
print()
print(df_bl)


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
for i, img_id in enumerate(test_ids[:4]):
    img, gt, pred, fov, _ = cache_bl[img_id]
    ovl = nakladka(img, pred, gt, fov)
    axes[i,0].imshow(img);              axes[i,0].set_title(img_id)
    axes[i,1].imshow(gt,  cmap='gray'); axes[i,1].set_title('Ground truth')
    axes[i,2].imshow(pred,cmap='gray'); axes[i,2].set_title('Predykcja Frangi')
    axes[i,3].imshow(ovl);              axes[i,3].set_title('TP=ziel FP=czer FN=nieb')
    for ax in axes[i]: ax.axis('off')
plt.suptitle('Wyniki baseline (Frangi)', fontsize=14)
plt.tight_layout(); plt.show()


---
## 2. Klasyfikator ML — wymagania na 4.0

Podejście:
- Wycinki **5×5 px** wokół każdego piksela → 13 cech (statystyki, momenty centralne, momenty Hu, vesselness)
- **Undersampling**: 1000 naczyń + 1000 tła na obraz → zbalansowany zbiór uczący
- **Random Forest** (100 drzew) — nie wymaga skalowania cech, daje feature importances
- **Hold-out**: train = Image_01–10, test = Image_11–14


In [ ]:
# ── ekstrakcja 13 cech z wycinków 5×5 ─────────────────────────────────────────
PATCH = 5
HALF  = PATCH // 2

# siatka współrzędnych wycinka (liczymy raz)
_xf = np.tile(np.arange(PATCH) - HALF, PATCH).astype(float)
_yf = np.repeat(np.arange(PATCH) - HALF, PATCH).astype(float)

def licz_cechy(patches, vess_vals):
    mu  = patches.mean(axis=1)
    dev = patches - mu[:, None]
    mu20 = (dev * _xf**2).mean(1); mu02 = (dev * _yf**2).mean(1)
    mu11 = (dev * _xf * _yf).mean(1)
    mu30 = (dev * _xf**3).mean(1); mu03 = (dev * _yf**3).mean(1)
    m00  = np.abs(patches.sum(1)); m00 = np.where(m00 < 1e-10, 1e-10, m00)
    e20, e02, e11 = mu20/m00**2, mu02/m00**2, mu11/m00**2
    H1 = e20 + e02
    H2 = (e20 - e02)**2 + 4*e11**2
    return np.column_stack([mu, patches.std(1), patches.min(1), patches.max(1),
                            patches.max(1)-patches.min(1),
                            mu20, mu02, mu11, mu30, mu03, H1, H2, vess_vals])

FEAT_NAMES = ["mean","std","min","max","range",
              "mu20","mu02","mu11","mu30","mu03","H1","H2","vesselness"]

def probki_rf(prep, vess, gt, fov, n_klas, rng):
    H, W = prep.shape
    pad   = np.pad(prep, HALF, 'reflect')
    gt_f  = gt.ravel(); fov_f = fov.ravel()
    pos   = np.where(fov_f & gt_f)[0]
    neg   = np.where(fov_f & ~gt_f)[0]
    sp    = rng.choice(pos, min(n_klas, len(pos)), replace=False)
    sn    = rng.choice(neg, min(n_klas, len(neg)), replace=False)
    sel   = np.concatenate([sp, sn])
    y     = np.concatenate([np.ones(len(sp)), np.zeros(len(sn))]).astype(int)
    r, c  = np.unravel_index(sel, (H, W))
    P     = np.zeros((len(sel), PATCH*PATCH), dtype=np.float64)
    for di in range(PATCH):
        for dj in range(PATCH):
            P[:, di*PATCH+dj] = pad[r+di, c+dj]
    return licz_cechy(P, vess.ravel()[sel]), y

def predykcja_rf(clf, prep, vess, fov):
    H, W  = prep.shape
    pad   = np.pad(prep, HALF, 'reflect')
    wins  = view_as_windows(pad, (PATCH, PATCH)).reshape(H*W, PATCH*PATCH)
    feats = licz_cechy(wins.astype(float), vess.ravel())
    return clf.predict(feats).reshape(H, W).astype(bool) & fov

print("Gotowe – 13 cech na wycinek 5×5")


In [ ]:
rng_rf = np.random.default_rng(42)
Xl, yl = [], []

for img_id in train_ids:
    img, gt = wczytaj(img_id)
    fov  = maska_fov(img)
    prep = preprocessing(img)
    vess = frangi(prep, sigmas=range(1, 6), black_ridges=False)
    Xi, yi = probki_rf(prep, vess, gt, fov, 1000, rng_rf)
    Xl.append(Xi); yl.append(yi)
    print(f"  {img_id}: {yi.sum()} naczyń + {(yi==0).sum()} tła")

X_train = np.vstack(Xl)
y_train = np.concatenate(yl)
print(f"\nDataset RF: {X_train.shape} | naczynia: {y_train.mean()*100:.1f}%")

clf_rf = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
clf_rf.fit(X_train, y_train)
print("Random Forest wytrenowany!")

imp = pd.Series(clf_rf.feature_importances_, index=FEAT_NAMES).sort_values(ascending=False)
print("\nNajważniejsze cechy:")
print(imp.round(4))


In [ ]:
wyniki_rf = []; cache_rf = {}

for img_id in test_ids:
    img, gt = wczytaj(img_id)
    fov  = maska_fov(img)
    prep = preprocessing(img)
    vess = frangi(prep, sigmas=range(1, 6), black_ridges=False)
    pred = predykcja_rf(clf_rf, prep, vess, fov)
    cache_rf[img_id] = pred
    m = ocen(pred, gt, fov); m['image'] = img_id
    wyniki_rf.append(m)
    print(f"  {img_id}: acc={m['accuracy']:.4f}  sens={m['sensitivity']:.4f}  spec={m['specificity']:.4f}")

df_rf = pd.DataFrame(wyniki_rf).set_index('image')[cols].round(4)
df_rf.loc["SREDNIA"] = df_rf.mean()

print("\n=== Porównanie: Frangi vs RF ===")
print(pd.DataFrame({"Frangi": df_bl.loc["SREDNIA"], "RF": df_rf.loc["SREDNIA"]}).round(4))


In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(22, 16))
for i, img_id in enumerate(test_ids[:4]):
    img, gt, pred_bl, fov, _ = cache_bl[img_id]
    pred_rf = cache_rf[img_id]
    for j, (im, cm, t) in enumerate([
        (img, None, img_id),
        (gt,      'gray', 'Ground truth'),
        (pred_bl, 'gray', 'Frangi'),
        (pred_rf, 'gray', 'RF'),
        (nakladka(img, pred_rf, gt, fov), None, 'RF nakładka'),
    ]):
        axes[i, j].imshow(im, cmap=cm)
        axes[i, j].set_title(t)
        axes[i, j].axis('off')
plt.suptitle('Frangi vs Random Forest  (TP=ziel FP=czer FN=nieb)', fontsize=13)
plt.tight_layout(); plt.show()


---
## 3. Głęboka sieć neuronowa (CNN) — wymagania na 5.0

Architektura CNN trenowanej na wycinkach **32×32 px**:
- 3 × blok Conv2D + BatchNorm + MaxPool (32 → 64 → 128 filtrów)
- GlobalAveragePooling zamiast Flatten (mniej parametrów, mniej przeuczenia)
- Dense(64) + Dropout(0.4) + Dense(1, sigmoid)

Zbiór uczący: te same obrazy co RF (Image_01–10), 500 naczyń + 500 tła na obraz.
Zbiór testowy (hold-out): Image_11–14 — identyczny z baseline i RF.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

PCNN = 32   # rozmiar wycinków dla CNN
HCNN = PCNN // 2

def zbuduj_cnn():
    model = keras.Sequential([
        layers.Input(shape=(PCNN, PCNN, 1)),

        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),

        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid'),
    ], name='vessel_cnn')

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

cnn = zbuduj_cnn()
cnn.summary()


In [ ]:
def probki_cnn(prep, gt, fov, n_klas, rng):
    H, W  = prep.shape
    pad   = np.pad(prep, HCNN, 'reflect')
    gt_f  = gt.ravel(); fov_f = fov.ravel()
    pos   = np.where(fov_f & gt_f)[0]
    neg   = np.where(fov_f & ~gt_f)[0]
    sp    = rng.choice(pos, min(n_klas, len(pos)), replace=False)
    sn    = rng.choice(neg, min(n_klas, len(neg)), replace=False)
    sel   = np.concatenate([sp, sn])
    y     = np.concatenate([np.ones(len(sp)), np.zeros(len(sn))]).astype(np.float32)
    r, c  = np.unravel_index(sel, (H, W))
    P     = np.zeros((len(sel), PCNN, PCNN, 1), dtype=np.float32)
    for di in range(PCNN):
        for dj in range(PCNN):
            P[:, di, dj, 0] = pad[r + di, c + dj]
    return P, y

rng_cnn = np.random.default_rng(7)
Xc, yc  = [], []

for img_id in train_ids:
    img, gt = wczytaj(img_id)
    fov     = maska_fov(img)
    prep    = preprocessing(img)
    Xi, yi  = probki_cnn(prep, gt, fov, 500, rng_cnn)
    Xc.append(Xi); yc.append(yi)
    print(f"  {img_id}")

X_cnn = np.concatenate(Xc)
y_cnn = np.concatenate(yc)
print(f"\nDataset CNN: {X_cnn.shape[0]} wycinków {X_cnn.shape[1]}x{X_cnn.shape[2]}")


In [ ]:
historia = cnn.fit(
    X_cnn, y_cnn,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# krzywe uczenia
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(historia.history['loss'],     label='train')
plt.plot(historia.history['val_loss'], label='val')
plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(historia.history['accuracy'],     label='train')
plt.plot(historia.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
def predykcja_cnn(model, prep, fov, batch_size=4096):
    H, W    = prep.shape
    pad     = np.pad(prep, HCNN, 'reflect')
    fov_idx = np.where(fov.ravel())[0]
    r, c    = np.unravel_index(fov_idx, (H, W))
    probs   = np.zeros(H * W, dtype=np.float32)

    for start in range(0, len(fov_idx), batch_size):
        end = min(start + batch_size, len(fov_idx))
        br, bc = r[start:end], c[start:end]
        P = np.zeros((end - start, PCNN, PCNN, 1), dtype=np.float32)
        for di in range(PCNN):
            for dj in range(PCNN):
                P[:, di, dj, 0] = pad[br + di, bc + dj]
        probs[fov_idx[start:end]] = model.predict(P, verbose=0).ravel()

    return (probs.reshape(H, W) > 0.5) & fov


In [ ]:
print("Predykcja CNN na zbiorze testowym (może chwilę potrwać)...")
wyniki_cnn = []; cache_cnn = {}

for img_id in test_ids:
    img, gt = wczytaj(img_id)
    fov  = maska_fov(img)
    prep = preprocessing(img)
    pred = predykcja_cnn(cnn, prep, fov)
    cache_cnn[img_id] = pred
    m = ocen(pred, gt, fov); m['image'] = img_id
    wyniki_cnn.append(m)
    print(f"  {img_id}: acc={m['accuracy']:.4f}  sens={m['sensitivity']:.4f}  spec={m['specificity']:.4f}")

df_cnn = pd.DataFrame(wyniki_cnn).set_index('image')[cols].round(4)
df_cnn.loc["SREDNIA"] = df_cnn.mean()

print("\n=== PORÓWNANIE WSZYSTKICH METOD (srednia z 8 obrazów) ===")
podsumowanie = pd.DataFrame({
    "Frangi (3.0)":        df_bl.loc["SREDNIA"],
    "RandomForest (4.0)":  df_rf.loc["SREDNIA"],
    "CNN (5.0)":           df_cnn.loc["SREDNIA"],
})
print(podsumowanie.round(4))


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for i, img_id in enumerate(test_ids[:4]):
    img, gt, pred_bl, fov, _ = cache_bl[img_id]
    pred_rf  = cache_rf[img_id]
    pred_cnn = cache_cnn[img_id]
    for j, (pred, t) in enumerate([
        (pred_bl,  'Frangi (3.0)'),
        (pred_rf,  'RF (4.0)'),
        (pred_cnn, 'CNN (5.0)'),
    ]):
        axes[i, j].imshow(nakladka(img, pred, gt, fov))
        axes[i, j].set_title(t)
        axes[i, j].axis('off')
    axes[i, 3].imshow(gt, cmap='gray')
    axes[i, 3].set_title('Ground truth')
    axes[i, 3].axis('off')
    axes[i, 0].set_ylabel(img_id, fontsize=9)
plt.suptitle('Porównanie metod  (TP=ziel FP=czer FN=nieb)', fontsize=13)
plt.tight_layout(); plt.show()
